In [1]:
"""
stage9_testset_map_evaluation.py
============================================================================
APPROACH A — evaluate uncertainty maps on the HELD-OUT TEST SPLIT.

WHY THIS RUN EXISTS
Stage 4 compared diffusion-ensemble variance against MC-Dropout variance and
MCD won nearly everywhere. That comparison was not like-for-like: the
DenseNet supplying the MCD maps had been trained supervised on those exact
images, while RoentGen was zero-shot. On the test split neither model has
seen the images, so the comparison becomes fair. That is the whole point of
Approach A -- it does not add a new method, it removes a confound.

UNIT OF ANALYSIS -- READ THIS BEFORE CHANGING ANYTHING
One AUROC per image, then bootstrap over IMAGES. Pooling every pixel from
every image into one ROC would give ~1.7e8 "samples", but pixels within an
image are heavily correlated and large-lesion images would dominate. The
resulting CI would be absurdly narrow and wrong. If the Stage 1 train-set
numbers were computed by pooling, recompute them this way before quoting
train and test side by side.

THREE METHODS, AND WHAT EACH ONE ACTUALLY IS
  diffusion     pixel-wise variance across N=20 stochastic reconstructions
                from a fixed inverted latent. Zero-shot: RoentGen never saw
                these images and never saw the labels.
  mcd_logit     MC-Dropout predictive variance from the DenseNet. Zero-shot
                w.r.t. these IMAGES on the test split, but the model was
                trained on this LABEL SET. Epistemic uncertainty.
  mcd_gradcam   Grad-CAM saliency from the same DenseNet.

  READ THE GRAD-CAM ROW WITH CARE. It is NOT an uncertainty estimate. It is
  a class-evidence attribution: it answers "which pixels drove the positive
  prediction", which is almost the definition of the localisation task being
  scored. Diffusion variance answers "where is the generative model
  unsure", which only coincides with the lesion if instability tracks
  pathology. Grad-CAM is therefore a SKYLINE -- an upper reference for what
  a label-supervised method achieves -- not a peer baseline. Reporting it as
  though diffusion "lost" to it misstates the comparison.

  Two further asymmetries to state in the thesis:
    - Both MCD variants come from a classifier trained with the very labels
      the boxes encode. The test split removes the image-level confound that
      broke Stage 4; it does NOT remove label supervision.
    - Grad-CAM maps originate at the last conv layer (~16x16) and are
      upsampled to 512. Large smooth blobs score well on box-overlap metrics
      almost mechanically. Compare the effective resolution of the three map
      types before attributing any gap to method quality.

METRICS, and why each is here
  pixel_auroc   ranking metric: does variance rank in-box pixels above
                out-of-box pixels? Threshold-free, prevalence-independent.
  pixel_auprc   paired with the in-box pixel fraction as the baseline. Boxes
                cover a small share of the image, so AUPRC must be read as
                lift over that fraction, never against 0.5.
  dice_at_p     Dice after binarising the map at its own P-th percentile.
                Threshold chosen from the MAP, not from the labels, so no
                leakage. Reported across several P since Dice is threshold-
                sensitive in a way AUROC is not.
  pointing_hit  does the single highest-variance pixel fall inside a box?
                Crude but robust to map scaling and to calibration, and it
                is the metric a clinician's intuition actually matches.

STATISTICS
  - Bootstrap over images (B=2000) for the CI on the mean.
  - Wilcoxon signed-rank against the 0.5 chance line for AUROC. Non-
    parametric because per-image AUROC is bounded and skewed.
  - Friedman test across all three methods on the images ALL of them cover.
    With three related samples, running three pairwise Wilcoxons alone
    inflates the error rate; Friedman is the omnibus that licenses the
    post-hoc pairs.
  - Post-hoc paired Wilcoxon on every method pair, BH-corrected across the
    whole pair x class x metric family.

CONTAMINATION LOGIC IS INVERTED HERE
Stage 7 asserts NO test image appears in any generated map. This script
asserts the opposite for its own inputs: these maps MUST be test images.
It also re-asserts that these maps were never used in Stage 6 training.
Both checks run; either failing invalidates the run.
"""

import json
import warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import zoom
from sklearn.metrics import roc_auc_score, average_precision_score
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# =============================================================================
# CONFIG — verify every path against the Kaggle sidebar before running
# =============================================================================

# One entry per class. Confirmed against the actual inspect_maps() output --
# every one of these directories was verified to contain chunked .npz files
# keyed "{image_id}__mean" / "{image_id}__variance" (diffusion files also
# carry "{image_id}__z_T", ignored via CHUNK_IGNORE_SUFFIXES above).
DIFFUSION_ROOT = "/kaggle/input/datasets/kartikichandratre"
MCD_LOGIT_ROOT = ("/kaggle/input/datasets/kartikichandratre/"
                  "stage4-mc-dropout-logit-testbaseline/stage4_mc_dropout_maps")
MCD_GRADCAM_ROOT = ("/kaggle/input/datasets/kartikichandratre/"
                    "stage4-mc-dropout-gradcam-testbaseline/stage4_mc_dropout_maps")

TEST_MAP_DIRS = {
    "Pneumothorax": {
        "diffusion":   f"{DIFFUSION_ROOT}/pneumothorax-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/pneumothorax_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/pneumothorax_gradcam"},
    "Consolidation": {
        "diffusion":   f"{DIFFUSION_ROOT}/consolidation-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/consolidation_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/consolidation_gradcam"},
    "Nodule/Mass": {
        "diffusion":   f"{DIFFUSION_ROOT}/nodule-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/nodule_mass_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/nodule_mass_gradcam"},
    "Cardiomegaly": {
        "diffusion":   f"{DIFFUSION_ROOT}/cardiomegaly-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/cardiomegaly_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/cardiomegaly_gradcam"},
    "Atelectasis": {
        "diffusion":   f"{DIFFUSION_ROOT}/atelectasis-uncertainty-test",
        "mcd_logit":   f"{MCD_LOGIT_ROOT}/atelectasis_logit",
        "mcd_gradcam": f"{MCD_GRADCAM_ROOT}/atelectasis_gradcam"},
}


# Plot and table ordering. Grad-CAM last because it is the skyline, not a peer.
METHOD_ORDER = ["diffusion", "mcd_logit", "mcd_gradcam"]
METHOD_COLORS = {"diffusion": "#3b6ea5", "mcd_logit": "#a5453b",
                 "mcd_gradcam": "#6b8f47"}

# Methods that are label-supervised. Reported, but never described as
# baselines that diffusion "should" beat.
SKYLINE_METHODS = {"mcd_gradcam"}

# Grad-CAM is signed in some implementations (ReLU'd in most). If your maps
# contain negatives and you want magnitude only, set True.
GRADCAM_ABS = False

# VinDr-CXR box annotations. Needs columns for image_id, class name, and
# x_min/y_min/x_max/y_max. Column names are remapped below if yours differ.
ANNOTATION_CSV = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv"
ANNOT_COLS = {"image_id": "image_id", "class_name": "class_name",
              "x_min": "x_min", "y_min": "y_min",
              "x_max": "x_max", "y_max": "y_max"}

METADATA_CSV_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"

# Directories whose ids went into Stage 6 TRAINING. Used to assert these
# test maps were never trained on.
STAGE6_TRAIN_MAP_DIRS = [
    "/kaggle/input/datasets/pgc17ms072/pneumothorax-uncertainity-train-control",
    "/kaggle/input/datasets/kartikichandratre/atelectasis-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/nodule-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/consolidation-gen-train",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-unceratainty-train-eta1-n20-g1-sess1",
    "/kaggle/input/datasets/kartikichandratre/cardiomelgaly-uncertainity-sess2",
]

# Boxes were drawn on the ORIGINAL DICOM resolution. Maps are at 512.
# If your annotation CSV is already rescaled to 512, set this False.
BOXES_IN_ORIGINAL_RESOLUTION = True
# width/height in ANNOTATION_CSV are named 'columns' and 'rows'
# respectively (image array dimensions), not 'original_width'/'height'.
ORIGINAL_SIZE_COLS = {"width": "columns", "height": "rows"}

MAP_SIZE = 512
DICE_PERCENTILES = [90.0, 95.0, 99.0]
N_BOOTSTRAP = 2000
ALPHA = 0.05
RANDOM_SEED = 42

OUTPUT_DIR = "/kaggle/working/stage9_testset_maps"

# Candidate array keys inside a SINGLE-MAP .npz. First match wins.
VARIANCE_KEYS = ["variance", "var", "pixel_variance", "sigma2",
                 "variance_map", "uncertainty", "std", "sigma",
                 # Grad-CAM / saliency naming
                 "cam", "gradcam", "grad_cam", "heatmap", "saliency",
                 "attribution", "map"]

# CHUNKED .npz support. Stage 4 writes many images per file, keyed
# "{image_id}__variance" and "{image_id}__mean". The separator and the
# suffixes that identify the array we want are configurable because the
# diffusion and Grad-CAM writers may not use the same convention.
CHUNK_SEPARATOR = "__"
CHUNK_VALUE_SUFFIXES = ["variance", "var", "uncertainty", "sigma2",
                        "cam", "gradcam", "heatmap", "saliency"]
# Suffixes present in the file but NOT to be scored (e.g. the ensemble mean).
CHUNK_IGNORE_SUFFIXES = ["mean", "std_err", "count", "n", "z_t"]


# =============================================================================
# STEP 0 — inspect. RUN THIS ALONE FIRST.
# =============================================================================
def inspect_maps(n_show=3):
    """Print what is actually inside the map files. Do this before anything."""
    for class_name, paths in TEST_MAP_DIRS.items():
        for kind in METHOD_ORDER:
            d = paths.get(kind)
            if d is None:
                print(f"{class_name:15s} {kind:12s} -- not provided")
                continue
            p = Path(d)
            if not p.exists():
                print(f"{class_name:15s} {kind:12s} -- MISSING DIR {d}")
                continue
            files = sorted(p.glob("*.npz")) + sorted(p.glob("*.npy"))
            print(f"\n{class_name:15s} {kind:12s} {len(files)} file(s) in {d}")
            for f in files[:n_show]:
                if f.suffix == ".npy":
                    a = np.load(f)
                    print(f"    {f.name}: npy {a.shape} {a.dtype}")
                    continue
                with np.load(f, allow_pickle=True) as z:
                    keys = list(z.files)
                    chunked = any(CHUNK_SEPARATOR in k for k in keys)
                    if chunked:
                        ids = sorted({k.rpartition(CHUNK_SEPARATOR)[0] for k in keys})
                        sufs = sorted({k.rpartition(CHUNK_SEPARATOR)[2].lower()
                                       for k in keys})
                        shape = z[keys[0]].shape
                        scored = [s for s in sufs if s in CHUNK_VALUE_SUFFIXES]
                        ignored = [s for s in sufs if s in CHUNK_IGNORE_SUFFIXES]
                        unknown = [s for s in sufs if s not in CHUNK_VALUE_SUFFIXES
                                   and s not in CHUNK_IGNORE_SUFFIXES]
                        print(f"    {f.name}: CHUNKED, {len(ids)} image(s), "
                              f"shape {shape}")
                        print(f"        suffixes  scored={scored}  "
                              f"ignored={ignored}  UNKNOWN={unknown}")
                        print(f"        example id: {ids[0]}")
                        if unknown:
                            print("        ^ decide whether any UNKNOWN suffix is "
                                  "the map, then update CHUNK_VALUE_SUFFIXES")
                    else:
                        print(f"    {f.name}: SINGLE-MAP, keys="
                              f"{ {k: z[k].shape for k in keys[:6]} }")
    print("\nWhen every row looks right, call main().")


# =============================================================================
# Loading
# =============================================================================
def _postprocess(m, method):
    """Reduce an ensemble stack if needed, apply Grad-CAM abs, resize."""
    if m.ndim == 3:
        warnings.warn(f"array is {m.shape}; taking per-pixel variance over axis 0.")
        m = m.var(axis=0)
    m = np.squeeze(m).astype(np.float64)
    if method == "mcd_gradcam" and GRADCAM_ABS:
        m = np.abs(m)
    if m.shape != (MAP_SIZE, MAP_SIZE):
        # Latent (64x64) and Grad-CAM (~16x16) maps upsample bilinearly.
        m = zoom(m, (MAP_SIZE / m.shape[0], MAP_SIZE / m.shape[1]), order=1)
    return m


def _read_chunked(z, path):
    """
    Chunked layout: many images per file, keys '{image_id}__{suffix}'.
    Returns {image_id: raw_array} for the value suffixes only, so the
    companion '__mean' arrays are never scored as if they were variance.
    """
    out, skipped = {}, set()
    for key in z.files:
        if CHUNK_SEPARATOR not in key:
            continue
        image_id, _, suffix = key.rpartition(CHUNK_SEPARATOR)
        s = suffix.lower()
        if s in CHUNK_IGNORE_SUFFIXES:
            continue
        if s not in CHUNK_VALUE_SUFFIXES:
            skipped.add(s)
            continue
        if image_id in out:
            raise KeyError(
                f"{path.name}: image_id '{image_id}' has more than one scorable "
                f"suffix. Narrow CHUNK_VALUE_SUFFIXES so exactly one matches."
            )
        out[image_id] = z[key]
    if skipped:
        warnings.warn(f"{path.name}: ignored unrecognised suffixes {sorted(skipped)}. "
                      f"Add to CHUNK_VALUE_SUFFIXES if one of them is the map.")
    return out


def _read_single(z, path):
    for k in VARIANCE_KEYS:
        if k in z.files:
            return {path.stem: z[k]}
    raise KeyError(
        f"{path.name} has keys {list(z.files)[:8]}, none in VARIANCE_KEYS and "
        f"none matching the chunked '{{id}}{CHUNK_SEPARATOR}{{suffix}}' pattern. "
        f"Run inspect_maps() and update the config."
    )


def load_map_dir(d, method=None):
    """image_id -> map at MAP_SIZE. Handles chunked and per-file layouts."""
    out, native = {}, []
    files = sorted(Path(d).glob("*.npz")) + sorted(Path(d).glob("*.npy"))
    for f in files:
        if f.suffix == ".npy":
            raw = {f.stem: np.load(f)}
        else:
            with np.load(f, allow_pickle=True) as z:
                chunked = any(CHUNK_SEPARATOR in k for k in z.files)
                raw = _read_chunked(z, f) if chunked else _read_single(z, f)
                raw = {k: np.array(v) for k, v in raw.items()}
        for image_id, arr in raw.items():
            if image_id in out:
                warnings.warn(f"duplicate image_id '{image_id}' in {d}; keeping first.")
                continue
            native.append(np.squeeze(arr).shape[0])
            out[image_id] = _postprocess(arr, method)
    return out, (float(np.median(native)) if native else np.nan)


def explode_boxes(df):
    """
    ANNOTATION_CSV is one row per IMAGE, with all boxes for that image
    packed into a 'boxes' column as a Python-literal string:
        "[{'class_name': 'Cardiomegaly', 'x_min': 863.0, ...}, {...}]"
    build_box_masks() expects one row per BOX (the ANNOT_COLS schema).
    This explodes wide -> long so the rest of the pipeline is unchanged.
    Images with no boxes, or an unparsable field, contribute nothing --
    correct, since there is no annotation to score against.
    """
    records = []
    for _, row in df.iterrows():
        raw = row.get("boxes")
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        try:
            boxes = literal_eval(raw) if isinstance(raw, str) else raw
        except (ValueError, SyntaxError):
            continue
        for b in boxes:
            records.append({
                "image_id": row["image_id"],
                "class_name": b.get("class_name"),
                "x_min": b.get("x_min"), "y_min": b.get("y_min"),
                "x_max": b.get("x_max"), "y_max": b.get("y_max"),
                "rad_id": b.get("rad_id"),
            })
    out = pd.DataFrame(records)
    n_classes = out["class_name"].nunique() if len(out) else 0
    print(f"  exploded {len(df)} image row(s) -> {len(out)} box row(s) "
          f"across {n_classes} classes")
    return out


def build_box_masks(annotations, metadata, class_name, image_ids):
    """image_id -> boolean mask of union of radiologist boxes for this class."""
    c = ANNOT_COLS
    sub = annotations[annotations[c["class_name"]] == class_name]
    sizes = {}
    if BOXES_IN_ORIGINAL_RESOLUTION:
        w, h = ORIGINAL_SIZE_COLS["width"], ORIGINAL_SIZE_COLS["height"]
        src = annotations if w in annotations.columns else metadata
        if w not in src.columns:
            raise KeyError(
                f"BOXES_IN_ORIGINAL_RESOLUTION=True but no '{w}' column in the "
                f"annotation or metadata CSV. Either add it or set the flag False."
            )
        sizes = {str(r[c["image_id"]] if c["image_id"] in src.columns else r["image_id"]):
                 (float(r[w]), float(r[h])) for _, r in src.iterrows()}

    masks = {}
    for image_id in image_ids:
        rows = sub[sub[c["image_id"]].astype(str) == str(image_id)]
        if rows.empty:
            continue
        mask = np.zeros((MAP_SIZE, MAP_SIZE), dtype=bool)
        sx = sy = 1.0
        if BOXES_IN_ORIGINAL_RESOLUTION:
            if str(image_id) not in sizes:
                continue
            ow, oh = sizes[str(image_id)]
            sx, sy = MAP_SIZE / ow, MAP_SIZE / oh
        for _, r in rows.iterrows():
            vals = [r[c["x_min"]], r[c["y_min"]], r[c["x_max"]], r[c["y_max"]]]
            if any(pd.isna(v) for v in vals):
                continue
            x0 = int(np.clip(round(r[c["x_min"]] * sx), 0, MAP_SIZE - 1))
            x1 = int(np.clip(round(r[c["x_max"]] * sx), 0, MAP_SIZE))
            y0 = int(np.clip(round(r[c["y_min"]] * sy), 0, MAP_SIZE - 1))
            y1 = int(np.clip(round(r[c["y_max"]] * sy), 0, MAP_SIZE))
            if x1 > x0 and y1 > y0:
                mask[y0:y1, x0:x1] = True
        if mask.any() and not mask.all():
            masks[str(image_id)] = mask
    return masks


# =============================================================================
# Per-image metrics
# =============================================================================
def dice(binary_map, mask):
    inter = np.logical_and(binary_map, mask).sum()
    denom = binary_map.sum() + mask.sum()
    return float(2 * inter / denom) if denom else np.nan


def metrics_for_image(var_map, mask):
    y = mask.ravel().astype(np.uint8)
    s = var_map.ravel().astype(np.float64)
    if not np.isfinite(s).all():
        s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
    if y.sum() == 0 or y.sum() == len(y):
        return None

    row = {
        "pixel_auroc": float(roc_auc_score(y, s)),
        "pixel_auprc": float(average_precision_score(y, s)),
        "box_fraction": float(y.mean()),
    }
    row["auprc_lift"] = row["pixel_auprc"] / row["box_fraction"]
    for p in DICE_PERCENTILES:
        row[f"dice_p{p:g}"] = dice(var_map >= np.percentile(var_map, p), mask)
    flat = int(np.argmax(s))
    row["pointing_hit"] = int(mask.ravel()[flat])
    return row


# =============================================================================
# Aggregation
# =============================================================================
def bootstrap_mean_ci(values, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    v = np.asarray([x for x in values if np.isfinite(x)], dtype=float)
    if len(v) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.array([rng.choice(v, size=len(v), replace=True).mean()
                      for _ in range(n_boot)])
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def paired_compare(per_image, class_name, metric, method_a, method_b):
    """Paired Wilcoxon on the images both methods cover."""
    sub = per_image[(per_image["class_name"] == class_name)]
    a_df = sub[sub["method"] == method_a][["image_id", metric]]
    b_df = sub[sub["method"] == method_b][["image_id", metric]]
    merged = a_df.merge(b_df, on="image_id", suffixes=("_a", "_b")).dropna()
    if len(merged) < 6:
        return None
    a, b = merged[f"{metric}_a"].to_numpy(), merged[f"{metric}_b"].to_numpy()
    if np.allclose(a, b):
        return None
    stat, p = stats.wilcoxon(a, b)
    mean_d, lo, hi = bootstrap_mean_ci(a - b)
    # Rank-biserial correlation: an effect size that survives non-normality.
    d = a - b
    nz = d[d != 0]
    rbc = float(np.sign(nz).mean()) if len(nz) else np.nan
    return {"class_name": class_name, "metric": metric,
            "method_a": method_a, "method_b": method_b,
            "n_paired": len(merged),
            "mean_a": float(a.mean()), "mean_b": float(b.mean()),
            "mean_delta": mean_d, "delta_ci_low": lo, "delta_ci_high": hi,
            "rank_biserial": rbc,
            "a_wins_frac": float((a > b).mean()),
            "wilcoxon_stat": float(stat), "p_wilcoxon": float(p),
            "a_is_skyline": method_a in SKYLINE_METHODS,
            "b_is_skyline": method_b in SKYLINE_METHODS}


def friedman_omnibus(per_image, class_name, metric, methods):
    """
    Friedman test on images covered by ALL methods. This is the omnibus that
    licenses the post-hoc pairwise tests; without it, three pairwise
    Wilcoxons per class inflate the family-wise error rate.
    """
    sub = per_image[per_image["class_name"] == class_name]
    wide = sub.pivot_table(index="image_id", columns="method", values=metric)
    present = [m for m in methods if m in wide.columns]
    if len(present) < 3:
        return None
    wide = wide[present].dropna()
    if len(wide) < 6:
        return None
    stat, p = stats.friedmanchisquare(*[wide[m].to_numpy() for m in present])
    row = {"class_name": class_name, "metric": metric,
           "n_complete_cases": len(wide), "methods": ",".join(present),
           "friedman_stat": float(stat), "p_friedman": float(p)}
    # Mean rank per method (1 = best), the natural Friedman effect summary.
    ranks = wide.rank(axis=1, ascending=False)
    for m in present:
        row[f"mean_rank_{m}"] = float(ranks[m].mean())
    return row


# =============================================================================
# Contamination checks — both directions
# =============================================================================
def assert_split_integrity(metadata, all_map_ids):
    test_ids = set(metadata[metadata["split"] == "test"]["image_id"].astype(str))
    train_ids = set(metadata[metadata["split"] == "train"]["image_id"].astype(str))

    not_test = all_map_ids - test_ids
    if not_test:
        raise AssertionError(
            f"{len(not_test)} map(s) are NOT test-split images, e.g. "
            f"{sorted(not_test)[:5]}. These maps must come from the held-out "
            f"split or Approach A proves nothing."
        )
    print(f"  FORWARD: all {len(all_map_ids)} maps are test-split images.")

    trained = set()
    for d in STAGE6_TRAIN_MAP_DIRS:
        p = Path(d)
        if not p.exists():
            print(f"  WARNING: Stage 6 train dir missing, cannot verify: {d}")
            continue
        trained |= {f.stem for f in p.glob("*.npz")} | {f.stem for f in p.glob("*.npy")}
    overlap = all_map_ids & trained
    if overlap:
        raise AssertionError(
            f"{len(overlap)} test map id(s) also appear in Stage 6 training "
            f"inputs, e.g. {sorted(overlap)[:5]}. Split is broken."
        )
    print(f"  REVERSE: no overlap with {len(trained)} Stage 6 training ids.")
    print(f"  (test split holds {len(test_ids)} images; train {len(train_ids)})")


# =============================================================================
# Figures
# =============================================================================
def plot_auroc_distributions(per_image, out_dir):
    classes = sorted(per_image["class_name"].unique())
    fig, ax = plt.subplots(figsize=(1.6 * len(classes) + 3, 4.4))
    data, labels, colors = [], [], []
    present = [m for m in METHOD_ORDER if m in set(per_image["method"])]
    for c in classes:
        for method in present:
            v = per_image[(per_image["class_name"] == c) &
                          (per_image["method"] == method)]["pixel_auroc"].dropna()
            if len(v):
                data.append(v.to_numpy())
                star = "*" if method in SKYLINE_METHODS else ""
                labels.append(f"{c}\n{method}{star}\nn={len(v)}")
                colors.append(METHOD_COLORS.get(method, "#888888"))
    bp = ax.boxplot(data, labels=labels, showmeans=True, patch_artist=True)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.55)
    ax.axhline(0.5, color="k", ls="--", lw=1.2, label="chance")
    ax.set_ylabel("Per-image pixel AUROC")
    ax.set_title("Held-out test split: does uncertainty localise the radiologist boxes?")
    ax.set_xlabel("* label-supervised skyline, not a peer baseline", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")
    plt.setp(ax.get_xticklabels(), fontsize=7)
    plt.tight_layout()
    plt.savefig(Path(out_dir) / "stage9_auroc_distributions.png", dpi=150, bbox_inches="tight")
    plt.close()


# =============================================================================
# Orchestration
# =============================================================================
def main():
    out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
    rows, summary_rows, paired_rows = [], [], []

    print("[1/5] Loading annotations and metadata...")
    annotations = pd.read_csv(ANNOTATION_CSV)
    annotations = explode_boxes(annotations)
    metadata = pd.read_csv(METADATA_CSV_PATH)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)

    print("[2/5] Loading maps...")
    loaded, native_res = {}, []
    all_ids = set()
    for class_name, paths in TEST_MAP_DIRS.items():
        loaded[class_name] = {}
        for method in METHOD_ORDER:
            d = paths.get(method)
            if d is None or not Path(d).exists():
                print(f"  {class_name:15s} {method:12s} -- MISSING {d}")
                continue
            maps, res = load_map_dir(d, method=method)
            if not maps:
                continue
            loaded[class_name][method] = maps
            all_ids |= set(maps.keys())
            native_res.append({"class_name": class_name, "method": method,
                               "n_maps": len(maps), "median_native_size": res})
            print(f"  {class_name:15s} {method:12s} {len(maps):4d} maps "
                  f"(native {res:g}px -> {MAP_SIZE}px)")
    if not all_ids:
        raise RuntimeError("No maps loaded. Run inspect_maps() and fix TEST_MAP_DIRS.")

    print("[3/5] Split integrity...")
    assert_split_integrity(metadata, all_ids)

    print("[4/5] Per-image metrics...")
    for class_name, methods in loaded.items():
        masks = build_box_masks(annotations, metadata, class_name,
                                set().union(*[set(m) for m in methods.values()]))
        print(f"  {class_name}: {len(masks)} image(s) with usable boxes")
        for method, maps in methods.items():
            for image_id, var_map in tqdm(maps.items(), desc=f"{class_name}/{method}",
                                          leave=False):
                if image_id not in masks:
                    continue
                m = metrics_for_image(var_map, masks[image_id])
                if m is None:
                    continue
                m.update({"class_name": class_name, "method": method,
                          "image_id": image_id})
                rows.append(m)

    per_image = pd.DataFrame(rows)
    if per_image.empty:
        raise RuntimeError(
            "No image matched a box mask. Most likely ANNOT_COLS, the class "
            "name spelling, or BOXES_IN_ORIGINAL_RESOLUTION is wrong."
        )
    per_image.to_csv(out / "stage9_per_image_metrics.csv", index=False)

    metric_cols = (["pixel_auroc", "pixel_auprc", "auprc_lift", "pointing_hit"]
                   + [f"dice_p{p:g}" for p in DICE_PERCENTILES])

    for (class_name, method), g in per_image.groupby(["class_name", "method"]):
        row = {"class_name": class_name, "method": method, "n_images": len(g),
               "mean_box_fraction": g["box_fraction"].mean()}
        for mc in metric_cols:
            mean, lo, hi = bootstrap_mean_ci(g[mc])
            row[f"{mc}_mean"], row[f"{mc}_ci_low"], row[f"{mc}_ci_high"] = mean, lo, hi
        v = g["pixel_auroc"].dropna().to_numpy()
        if len(v) >= 6:
            _, p = stats.wilcoxon(v - 0.5)
            row["p_vs_chance"] = float(p)
            row["frac_above_chance"] = float((v > 0.5).mean())
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    ok = summary["p_vs_chance"].notna()
    summary.loc[ok, "p_vs_chance_bh"] = multipletests(
        summary.loc[ok, "p_vs_chance"], alpha=ALPHA, method="fdr_bh")[1]
    summary.to_csv(out / "stage9_summary.csv", index=False)

    print("\n" + "=" * 78)
    print("HELD-OUT TEST SPLIT — localisation")
    print("=" * 78)
    print(summary[["class_name", "method", "n_images", "pixel_auroc_mean",
                   "pixel_auroc_ci_low", "pixel_auroc_ci_high",
                   "pointing_hit_mean", "p_vs_chance_bh"]].round(4).to_string(index=False))

    # Resolution audit -- the confound that can explain a Grad-CAM advantage.
    res_df = pd.DataFrame(native_res)
    res_df.to_csv(out / "stage9_native_resolutions.csv", index=False)
    print("\nNative map resolution before upsampling (read alongside any gap):")
    print(res_df.pivot_table(index="class_name", columns="method",
                             values="median_native_size").to_string())

    print("\n[5/6] Friedman omnibus across the three methods...")
    omni_rows = []
    for class_name in loaded:
        for mc in ["pixel_auroc", "pointing_hit"]:
            r = friedman_omnibus(per_image, class_name, mc, METHOD_ORDER)
            if r:
                omni_rows.append(r)
    if omni_rows:
        omni = pd.DataFrame(omni_rows)
        omni["p_friedman_bh"] = multipletests(omni["p_friedman"], alpha=ALPHA,
                                              method="fdr_bh")[1]
        omni.to_csv(out / "stage9_friedman_omnibus.csv", index=False)
        cols = ["class_name", "metric", "n_complete_cases", "friedman_stat",
                "p_friedman_bh"] + [f"mean_rank_{m}" for m in METHOD_ORDER
                                    if f"mean_rank_{m}" in omni.columns]
        print(omni[cols].round(4).to_string(index=False))
        print("  mean_rank: 1 = best of the three on that image. ")
    else:
        print("  Skipped -- no class has all three methods on >=6 shared images.")

    print("\n[6/6] Post-hoc pairwise comparisons...")
    method_pairs = [("diffusion", "mcd_logit"),
                    ("diffusion", "mcd_gradcam"),
                    ("mcd_logit", "mcd_gradcam")]
    for class_name, methods in loaded.items():
        for ma, mb in method_pairs:
            if ma not in methods or mb not in methods:
                continue
            for mc in ["pixel_auroc", "pointing_hit"]:
                r = paired_compare(per_image, class_name, mc, ma, mb)
                if r:
                    paired_rows.append(r)

    if paired_rows:
        paired = pd.DataFrame(paired_rows)
        paired["p_bh"] = multipletests(paired["p_wilcoxon"], alpha=ALPHA,
                                       method="fdr_bh")[1]
        paired["comparison"] = paired["method_a"] + " vs " + paired["method_b"]
        paired.to_csv(out / "stage9_pairwise_comparisons.csv", index=False)
        for mc in ["pixel_auroc", "pointing_hit"]:
            g = paired[paired["metric"] == mc]
            if g.empty:
                continue
            print(f"\n  --- {mc} ---")
            print(g[["class_name", "comparison", "n_paired", "mean_a", "mean_b",
                     "mean_delta", "delta_ci_low", "delta_ci_high",
                     "rank_biserial", "p_bh"]].round(4).to_string(index=False))

        print("\nINTERPRETATION GUARD")
        print("  diffusion vs mcd_logit   -- the fair comparison. Both are")
        print("      uncertainty estimates; on this split neither model has")
        print("      seen these images. This is what Stage 4 could not do.")
        print("  diffusion vs mcd_gradcam -- NOT a fair comparison. Grad-CAM is")
        print("      class-evidence attribution from a label-supervised model,")
        print("      i.e. it is optimised for the very thing being scored.")
        print("      Report it as a skyline. A gap here is expected and is not")
        print("      evidence that diffusion uncertainty failed.")
        print("  Check the resolution table above before reading any gap as")
        print("      method quality: coarse upsampled maps flatter box overlap.")
    else:
        print("  No class has two or more methods -- skipped.")

    plot_auroc_distributions(per_image, out)
    print(f"\nSaved to {out}")
    print("\nNOTES FOR THE WRITE-UP")
    print("  1. Metrics are per-image, bootstrapped over images. If the Stage 1")
    print("     train numbers pooled pixels, recompute them this way before")
    print("     quoting train and test together.")
    print("  2. The test split removes the IMAGE-level confound from Stage 4.")
    print("     It does not remove LABEL supervision: both MCD variants come")
    print("     from a classifier trained on these classes. Diffusion saw")
    print("     neither the images nor the labels.")
    print("  3. These CIs cover image sampling only. The maps are themselves")
    print("     stochastic (N=20 reconstructions); a second generation seed")
    print("     would move them by an unmeasured amount.")
    return per_image, summary


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    inspect_maps()
    print("\nInspection done. Call main() once VARIANCE_KEYS and paths are set.")


Pneumothorax    diffusion    1 file(s) in /kaggle/input/datasets/kartikichandratre/pneumothorax-uncertainty-test
    uncertainty_chunk_000.npz: CHUNKED, 18 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean', 'z_t']  UNKNOWN=[]
        example id: 05d676834dbed1639cb5eea70c1e307b

Pneumothorax    mcd_logit    1 file(s) in /kaggle/input/datasets/kartikichandratre/stage4-mc-dropout-logit-testbaseline/stage4_mc_dropout_maps/pneumothorax_logit
    uncertainty_chunk_000.npz: CHUNKED, 18 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean']  UNKNOWN=[]
        example id: 05d676834dbed1639cb5eea70c1e307b

Pneumothorax    mcd_gradcam  1 file(s) in /kaggle/input/datasets/kartikichandratre/stage4-mc-dropout-gradcam-testbaseline/stage4_mc_dropout_maps/pneumothorax_gradcam
    uncertainty_chunk_000.npz: CHUNKED, 18 image(s), shape (512, 512)
        suffixes  scored=['variance']  ignored=['mean']  UNKNOWN=[]
        example id: 05d676834

In [2]:
main()

[1/5] Loading annotations and metadata...
  exploded 660 image row(s) -> 5563 box row(s) across 14 classes
[2/5] Loading maps...
  Pneumothorax    diffusion      18 maps (native 512px -> 512px)
  Pneumothorax    mcd_logit      18 maps (native 512px -> 512px)
  Pneumothorax    mcd_gradcam    18 maps (native 512px -> 512px)
  Consolidation   diffusion      54 maps (native 512px -> 512px)
  Consolidation   mcd_logit      54 maps (native 512px -> 512px)
  Consolidation   mcd_gradcam    54 maps (native 512px -> 512px)
  Nodule/Mass     diffusion     117 maps (native 512px -> 512px)
  Nodule/Mass     mcd_logit     117 maps (native 512px -> 512px)
  Nodule/Mass     mcd_gradcam   117 maps (native 512px -> 512px)
  Cardiomegaly    diffusion     331 maps (native 512px -> 512px)
  Cardiomegaly    mcd_logit     331 maps (native 512px -> 512px)
  Cardiomegaly    mcd_gradcam   331 maps (native 512px -> 512px)
  Atelectasis     diffusion      27 maps (native 512px -> 512px)
  Atelectasis     mcd_logi

Pneumothorax/diffusion:   0%|          | 0/18 [00:00<?, ?it/s]

Pneumothorax/mcd_logit:   0%|          | 0/18 [00:00<?, ?it/s]

Pneumothorax/mcd_gradcam:   0%|          | 0/18 [00:00<?, ?it/s]

  Consolidation: 54 image(s) with usable boxes


...

Atelectasis/mcd_gradcam:   0%|          | 0/27 [00:00<?, ?it/s]


HELD-OUT TEST SPLIT — localisation
   class_name      method  n_images  pixel_auroc_mean  pixel_auroc_ci_low  pixel_auroc_ci_high  pointing_hit_mean  p_vs_chance_bh
  Atelectasis   diffusion        27            0.6348              0.5987               0.6720             0.0000          0.0000
  Atelectasis mcd_gradcam        27            0.4212              0.3546               0.4866             0.0000          0.0265
  Atelectasis   mcd_logit        27            0.7399              0.6807               0.7920             0.0741          0.0000
 Cardiomegaly   diffusion       331            0.4502              0.4420               0.4586             0.0000          0.0000
 Cardiomegaly mcd_gradcam       331            0.6003              0.5903               0.6102             0.2085          0.0000
 Cardiomegaly   mcd_logit       331            0.7551              0.7440               0.7655             0.2779          0.0000
Consolidation   diffusion        54            0.6939 

/tmp/ipykernel_58/3087213688.py:552: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=labels, showmeans=True, patch_artist=True)



Saved to /kaggle/working/stage9_testset_maps

NOTES FOR THE WRITE-UP
  1. Metrics are per-image, bootstrapped over images. If the Stage 1
     train numbers pooled pixels, recompute them this way before
     quoting train and test together.
  2. The test split removes the IMAGE-level confound from Stage 4.
     It does not remove LABEL supervision: both MCD variants come
     from a classifier trained on these classes. Diffusion saw
     neither the images nor the labels.
  3. These CIs cover image sampling only. The maps are themselves
     stochastic (N=20 reconstructions); a second generation seed
     would move them by an unmeasured amount.


(      pixel_auroc  pixel_auprc  box_fraction  auprc_lift  dice_p90  dice_p95  \
 0        0.478536     0.160115      0.185108    0.864980  0.000000  0.000000   
 1        0.748274     0.082480      0.045063    1.830321  0.075891  0.000000   
 2        0.226384     0.002257      0.003693    0.611294  0.000000  0.000000   
 3        0.556860     0.148785      0.150566    0.988173  0.012758  0.000799   
 4        0.547235     0.045144      0.045807    0.985524  0.000576  0.000000   
 ...           ...          ...           ...         ...       ...       ...   
 1636     0.490813     0.221046      0.232338    0.951398  0.122726  0.064366   
 1637     0.329416     0.016318      0.024353    0.670043  0.007792  0.002873   
 1638     0.432398     0.061199      0.071083    0.860946  0.057482  0.047571   
 1639     0.140373     0.006701      0.012199    0.549311  0.000000  0.000000   
 1640     0.623047     0.111816      0.083942    1.332060  0.114019  0.063623   
 
       dice_p99  pointing_

In [9]:
"""
plot_variance_heatmap_examples.py
============================================================================
Figure 5.1 -- variance_heatmap_examples.png

Qualitative grid: one row per pathology class, three columns --
  (1) the input radiograph
  (2) the pixel-wise variance heatmap
  (3) the heatmap with radiologist bounding boxes overlaid

PASTE AS A CELL *AFTER* stage9_final.py, in the same notebook. This reuses
its already-defined TEST_MAP_DIRS, explode_boxes(), build_box_masks(),
load_map_dir(), MAP_SIZE, ANNOTATION_CSV and METADATA_CSV_PATH rather than
redefining them -- if those aren't in scope, run stage9_final.py's cell
first (inspect_maps() is enough; you don't need main() to have finished).

WHAT THIS SCRIPT ADDS THAT stage9_final.py DOESN'T HAVE
The raw radiograph pixel array. stage9_final.py only ever touches
uncertainty maps and box coordinates -- it never loads the image itself.
That lives in the H5 archive described in Methodology (Section 3.2):
"Images are converted from DICOM, windowed, resized to a common resolution
and stored in a single HDF5 archive." This script's first job is
INSPECT_H5(), which prints the archive's actual key structure so you can
confirm H5_IMAGE_KEY_FN below before trusting anything downstream --
I have not seen this file and am guessing the common `h5file[image_id]`
convention. If your archive nests images under a group, or keys by
something other than the raw image_id string, this will need a one-line
fix after you look at the inspection output.

EXAMPLE SELECTION
One test image per class, chosen as the image with the LARGEST annotated
box area for that class -- this maximises the chance the qualitative
pattern (or its absence, for Cardiomegaly) is visible at figure scale
rather than picking an image where the finding is a few pixels wide.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import h5py

# =============================================================================
# CONFIG
# =============================================================================
H5_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"  # verify via inspect_h5()

# How to pull an image array out of the open h5py.File given an image_id.
# Default assumes h5file[image_id] returns the array directly. If
# inspect_h5() shows a different layout (e.g. a top-level "images" group),
# fix this one function and nothing else needs to change.
def h5_get_image(h5file, image_id):
    return h5file[image_id][()]

FIGURE_CLASSES = ["Pneumothorax", "Consolidation", "Nodule/Mass",
                  "Cardiomegaly", "Atelectasis"]

OUTPUT_PNG = "/kaggle/working/variance_heatmap_examples.png"


# =============================================================================
# STEP 0 -- inspect. Run this alone first.
# =============================================================================
def inspect_h5():
    with h5py.File(H5_PATH, "r") as f:
        keys = list(f.keys())
        print(f"{len(keys)} top-level key(s) in {H5_PATH}")
        print("first 5 keys:", keys[:5])
        sample = f[keys[0]]
        if isinstance(sample, h5py.Dataset):
            print(f"'{keys[0]}' is a Dataset, shape={sample.shape}, dtype={sample.dtype}")
            print("-> h5_get_image() default (h5file[image_id]) should work as-is.")
        else:
            print(f"'{keys[0]}' is a Group, sub-keys: {list(sample.keys())[:5]}")
            print("-> h5_get_image() needs updating to descend into this group.")


# =============================================================================
# Example selection: largest box per class
# =============================================================================
def select_example_images(annotations_long, target_percentile=60):
    """
    Per class, the box whose area is closest to the target percentile of
    that class's area distribution -- NOT the largest.

    Max-area selection was tried first and rejected: it reliably surfaces
    VinDr's known outlier annotations (e.g. a Pneumothorax box spanning
    77% of image height, far larger than the thin pleural-rim finding the
    class actually represents). A percentile pick still gives a box large
    enough to read at figure scale without cherry-picking the dataset's
    most atypical example of each class.
    """
    df = annotations_long.copy()
    df["area"] = (df["x_max"] - df["x_min"]) * (df["y_max"] - df["y_min"])
    chosen = {}
    for class_name in FIGURE_CLASSES:
        sub = df[df["class_name"] == class_name]
        if sub.empty:
            print(f"  WARNING: no boxes found for {class_name}, skipping")
            continue
        target_area = np.percentile(sub["area"], target_percentile)
        row = sub.iloc[(sub["area"] - target_area).abs().argsort().iloc[0]]
        chosen[class_name] = row["image_id"]
        print(f"  {class_name:15s} -> {row['image_id']} "
              f"(box area {row['area']:.0f} px^2, "
              f"class p{target_percentile}={target_area:.0f} px^2)")
    return chosen


# =============================================================================
# Figure
# =============================================================================
def _scaled_box_coords(annotations_long, metadata, class_name, image_id, target_shape):
    """
    Return box coordinates rescaled to target_shape (h, w), plus the raw
    native coordinates and the native image size used for scaling -- all
    returned together so the caller can print them for debugging.
    """
    rows = annotations_long[
        (annotations_long["class_name"] == class_name)
        & (annotations_long["image_id"].astype(str) == str(image_id))
    ]
    meta_row = metadata[metadata["image_id"].astype(str) == str(image_id)]
    if meta_row.empty:
        return None, rows, None
    ow, oh = float(meta_row.iloc[0]["columns"]), float(meta_row.iloc[0]["rows"])
    th, tw = target_shape
    sx, sy = tw / ow, th / oh

    boxes_scaled = []
    for _, r in rows.iterrows():
        boxes_scaled.append((r["x_min"] * sx, r["y_min"] * sy,
                            r["x_max"] * sx, r["y_max"] * sy))
    return boxes_scaled, rows, (ow, oh, sx, sy)


def plot_examples(chosen, annotations_long, metadata):
    n = len(chosen)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3.15 * n))
    if n == 1:
        axes = axes.reshape(1, 3)

    with h5py.File(H5_PATH, "r") as h5file:
        for i, (class_name, image_id) in enumerate(chosen.items()):
            img = h5_get_image(h5file, image_id).astype(np.float32)
            if img.max() > 1.5:  # normalise for display if stored as 0-255
                img = img / 255.0

            maps, _ = load_map_dir(TEST_MAP_DIRS[class_name]["diffusion"],
                                    method="diffusion")
            if image_id not in maps:
                print(f"  WARNING: {image_id} not found in {class_name} maps -- skipping row")
                continue
            var_map = maps[image_id]

            # Scaling confirmed correct against the rendered figure -- see
            # the diagnostic version of this function in project history if
            # this ever needs re-verifying against a different H5 archive.
            boxes_on_img, raw_rows, _ = _scaled_box_coords(
                annotations_long, metadata, class_name, image_id, img.shape[:2])
            boxes_on_var, _, _ = _scaled_box_coords(
                annotations_long, metadata, class_name, image_id, var_map.shape[:2])

            n_readers = len(raw_rows)
            reader_note = f" ({n_readers} readers)" if n_readers > 1 else ""

            ax_img, ax_var, ax_overlay = axes[i]

            ax_img.imshow(img, cmap="gray")
            for (x0, y0, x1, y1) in (boxes_on_img or []):
                ax_img.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                              fill=False, edgecolor="cyan", lw=1.2))
            ax_img.set_title(f"{class_name}{reader_note}\ninput", fontsize=9)
            ax_img.axis("off")

            ax_var.imshow(var_map, cmap="inferno")
            ax_var.set_title("variance", fontsize=9)
            ax_var.axis("off")

            ax_overlay.imshow(var_map, cmap="inferno")
            for (x0, y0, x1, y1) in (boxes_on_var or []):
                ax_overlay.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                                   fill=False, edgecolor="cyan", lw=1.2))
            ax_overlay.set_title("variance + box", fontsize=9)
            ax_overlay.axis("off")

    fig.suptitle("Representative pixel-wise variance maps", fontsize=13, y=0.995)
    fig.text(0.5, 0.975,
             "Variance concentrates along anatomical edges (diaphragm, rib margins) "
             "regardless of pathology class -- the visual basis of the dose--response "
             "and pointing-hit results in Section 5.1--5.2. Multiple boxes on a row "
             "reflect independent radiologist annotations of the same finding, not a "
             "rendering artefact.",
             ha="center", va="top", fontsize=8, style="italic", wrap=True,
             transform=fig.transFigure)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nsaved {OUTPUT_PNG}")


# =============================================================================
# Orchestration
# =============================================================================
def main():
    print("[1/4] Loading annotations and metadata...")
    annotations = pd.read_csv(ANNOTATION_CSV)
    annotations_long = explode_boxes(annotations)
    metadata = pd.read_csv(METADATA_CSV_PATH)

    print("\n[2/4] Selecting example images (largest box per class)...")
    chosen = select_example_images(annotations_long)

    print("\n[3/4] Building figure...")
    plot_examples(chosen, annotations_long, metadata)

    print("\n[4/4] Done.")


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    inspect_h5()
    print("\nInspection done. Verify H5_PATH and h5_get_image() above, "
          "then call main().")

4394 top-level key(s) in /kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5
first 5 keys: ['0005e8e3701dfb1dd93d53e2ff537b6e', '0007d316f756b3fa0baea2ff514ce945', '000d68e42b71d3eac10ccc077aba07c1', '00150343289f317a0ad5629d5b7d9ef9', '001d127bad87592efe45a5c7678f8b8d']
'0005e8e3701dfb1dd93d53e2ff537b6e' is a Dataset, shape=(512, 512), dtype=float32
-> h5_get_image() default (h5file[image_id]) should work as-is.

Inspection done. Verify H5_PATH and h5_get_image() above, then call main().


In [10]:
main()

[1/4] Loading annotations and metadata...
  exploded 660 image row(s) -> 5563 box row(s) across 14 classes

[2/4] Selecting example images (largest box per class)...
  Pneumothorax    -> 7a9bbc3d02750c716fa773dcae80363b (box area 583869 px^2, class p60=588715 px^2)
  Consolidation   -> d59d5dcc1601a29509f91dab5f8550bc (box area 232245 px^2, class p60=224355 px^2)
  Nodule/Mass     -> fc50039c45fdb6c9224bfff5ba4e64b3 (box area 23142 px^2, class p60=23368 px^2)
  Cardiomegaly    -> 4a24da485b9550c8df8b19caff945cdc (box area 425250 px^2, class p60=425248 px^2)
  Atelectasis     -> 3c63e58fcda26e02fdd6619515399985 (box area 365190 px^2, class p60=350337 px^2)

[3/4] Building figure...

saved /kaggle/working/variance_heatmap_examples.png

[4/4] Done.
